In [0]:
df = spark.table('workspace.csv.btc_usd_ytd')

# Convert to pandas
pdf = df.toPandas()

# Change snapped_at to date

pdf['snapped_at'] = pdf['snapped_at'].dt.date

# Rename snapped_at
pdf.rename(columns={'snapped_at': 'date'}, inplace=True)
pdf.drop(columns=['total_volume'], inplace=True)

# add target column
pdf['target'] = pdf['price'].shift(-1)
pdf.dropna(inplace=True)

pdf=pdf.sort_values(by='date')

display(pdf)

date,price,market_cap,target
2013-04-28,135.3,1.50051759E9,141.96
2013-04-29,141.96,1.575032004E9,135.3
2013-04-30,135.3,1.501657493E9,117.0
2013-05-01,117.0,1.29895155E9,103.43
2013-05-02,103.43,1.148667722E9,91.01
2013-05-03,91.01,1.011066494E9,111.25
2013-05-04,111.25,1.236351844E9,116.79
2013-05-05,116.79,1.298377788E9,118.33
2013-05-06,118.33,1.315992304E9,106.4
2013-05-07,106.4,1.1837665E9,112.64


In [0]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Features and Target
features = ['price', 'market_cap']
x = pdf[features]
y = pdf['target']

#Split data
x_train, x_test, y_train, y_test = train_test_split(
    x,y,
    shuffle = False,
    test_size=0.2
)

#Train model
model = LinearRegression()
model.fit(x_train, y_train)

#Predict
Predictions = model.predict(x_test)

In [0]:
import pandas as pd

x_test_reset = x_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

date_col = pdf.loc[x_test.index, 'date'].reset_index(drop=True)

#Build results table

results = pd.DataFrame({
    'date': date_col,
    'price': x_test_reset['price'].round(2),
    'Predicted Price': Predictions.round(2)
})

#Difference
results['difference'] = (results['Predicted Price'] - results['price']).round(2)
results['% Error'] = ((results['difference']/results['price'])*100).round(2)
#Results 
display(results)


date,price,Predicted Price,difference,% Error
2023-01-31,22840.39,22813.96,-26.43,-0.12
2023-02-01,23137.32,23108.82,-28.5,-0.12
2023-02-02,23725.16,23695.55,-29.61,-0.12
2023-02-03,23539.68,23510.77,-28.91,-0.12
2023-02-04,23451.58,23422.11,-29.47,-0.13
2023-02-05,23340.35,23311.03,-29.32,-0.13
2023-02-06,22946.29,22917.11,-29.18,-0.13
2023-02-07,22786.48,22759.39,-27.09,-0.12
2023-02-08,23294.91,23265.56,-29.35,-0.13
2023-02-09,22947.51,22919.03,-28.48,-0.12
